# Section 07c Onboarding Notebook: Theory and Simulation of Polarons in the Weak and Strong Coupling Regimes

### Overview & Pedagogical Design
When an excess electron, hole, or exciton moves through a deformable crystal, molecular solid, polar semiconductor, or halide perovskite, its Coulomb field displaces the surrounding nuclei. The charge carrier dressed by its accompanying cloud of virtual phonons forms a quasiparticle known as a **polaron**. Depending on the competition between the electronic kinetic energy (half-bandwidth $W_0 = 2dJ$ or band mass $m_b$), the phonon energy $\hbar\omega_0$, and the electron–phonon (e–ph) relaxation energy $E_p$, polarons exhibit two qualitatively distinct physical regimes ([Phys. Rev. B 104, 235123 (2021)](https://link.aps.org/doi/10.1103/PhysRevB.104.235123); [Phys. Rev. X 10, 021062 (2020)](https://link.aps.org/doi/10.1103/PhysRevX.10.021062); [arXiv:2603.07290](https://arxiv.org/html/2603.07290v2)):

| Physical Feature | **Weak-Coupling (Large / Fröhlich) Regime** ($\alpha \ll 6$ or $\lambda, g^2 \ll 1$) | **Strong-Coupling (Small / Holstein–Lang–Firsov) Regime** ($\alpha \gg 6$ or $\lambda, g^2 \gg 1$) |
| :--- | :--- | :--- |
| **Spatial Extent ($a_P$)** | Delocalized over many unit cells ($a_P \gg a_{\text{lat}}$) | Self-trapped on a single site or bond ($a_P \lesssim a_{\text{lat}}$) |
| **Analytical Framework** | 2nd-order Rayleigh–Schrödinger Perturbation Theory / Redfield | **Lang–Firsov Canonical Transformation** $e^{\hat{S}}$ / Landau–Pekar Variational Ansatz |
| **Ground-State Energy Shift** | Linear in coupling: $\Delta E \approx -\alpha\hbar\omega_{\text{LO}}$ (or $-g^2\omega_0 \frac{J+\omega_0}{2J+\omega_0}$) | Quadratic in $g$: $E_{\text{pol}} \approx -E_p = -g^2\hbar\omega_0$ (or $-\frac{\alpha^2}{3\pi}\hbar\omega_{\text{LO}}$) |
| **Effective Mass ($m^*/m_0$)** | Mild perturbative enhancement: $m^*/m_b \approx (1 - \alpha/6)^{-1}$ | Exponential Franck–Condon mass enhancement: $m^*/m_0 = J/\tilde{J} = e^{g^2 \coth(\hbar\omega_0/2k_B T)}$ |
| **Spectral Function $A(\omega)$** | Sharp quasiparticle peak ($Z \approx 1 - \alpha/2$) + weak continuum | Suppressed zero-phonon line ($Z_0 = e^{-g^2}$) + Poisson progression of phonon replicas |
| **Charge Transport Mechanism** | Coherent band-like motion ($\frac{d\mu}{dT} < 0$, phonon-scattering limited) | Incoherent thermally activated Marcus hopping ($\mu \propto T^{-3/2}e^{-E_a/k_B T}$, $\frac{d\mu}{dT} > 0$) |

In this notebook, **no polaron theory or simulation functions are pre-solved for you**. You will implement **15 granular `# TODO` functions** covering both continuum and lattice polaron models across all coupling regimes:
- **Part 1 — Continuum Fröhlich Polarons: Weak-Coupling Perturbation Theory vs. Strong-Coupling Landau–Pekar Theory (`TODO 1` – `TODO 3`)**
- **Part 2 — Lattice Holstein Model: Exact Diagonalization Across the Weak-to-Strong Coupling Crossover (`TODO 4` – `TODO 7`)**
- **Part 3 — Strong-Coupling Lang–Firsov Canonical Transformation, Band Narrowing & Poisson Spectral Replicas (`TODO 8` – `TODO 11`)**
- **Part 4 — Real-Time Polaron Quantum Dynamics, Coherent-to-Incoherent Transport Crossover & Ehrenfest Self-Trapping (`TODO 12` – `TODO 15`)**
- **Part 5 — Step-by-Step Checkpoint & Verification Suite (`run_section07b_polaron_checkpoints`)**

In [ ]:
import numpy as np
from scipy.linalg import eigh, expm
from scipy.special import factorial
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True, linewidth=110)
HBAR_EV_S = 6.582119569e-16
M_ELEC_SI = 9.1093837015e-31
EV_TO_J = 1.602176634e-19
RYDBERG_EV = 13.605693122994
BOHR_TO_ANG = 0.529177210903
print("Section 07b Polaron Theory & Simulation environment initialized. Implement TODOs 1-15 and run `run_section07b_polaron_checkpoints()`!")

## Part 1: Continuum Fröhlich Polarons — Weak-Coupling Perturbation Theory vs. Strong-Coupling Landau–Pekar Theory (TODOs 1–3)

### Mathematical Specifications
In polar crystals (such as $\text{GaAs}$, $\text{NaCl}$, $\text{SrTiO}_3$, and lead-halide perovskites $\text{MAPbI}_3$), a conduction electron with bare band effective mass $m_b$ (in units of free electron mass $m_e$) interacts macroscopically with longitudinal optical (LO) phonons of energy $\hbar\omega_{\text{LO}}$ via the **Fröhlich Hamiltonian** ([Phys. Rev. B 104, 235123 (2021)](https://link.aps.org/doi/10.1103/PhysRevB.104.235123); [arXiv:0904.3682](https://arxiv.org/html/0904.3682v2)):

1. **Dimensionless Fröhlich Coupling Constant $\alpha$ & Polaron Length Scale $r_0$ (`compute_frohlich_alpha_and_length` — TODO 1)**:
   Using the Rydberg energy $E_{\text{Ry}} = \frac{m_e e^4}{2(4\pi\varepsilon_{\text{vac}}\hbar)^2} = 13.605693\text{ eV}$ and Bohr radius $a_0 = 0.529177\text{ \AA}$, the dimensionless Fröhlich constant $\alpha$ and characteristic polaron length $r_0 = \sqrt{\frac{\hbar}{2 m_b m_e \omega_{\text{LO}}}}$ (in $\text{\AA}$) take the exact form:
   $$\alpha = \left(\frac{1}{\varepsilon_\infty} - \frac{1}{\varepsilon_0}\right)\sqrt{\frac{m_b\,E_{\text{Ry}}}{\hbar\omega_{\text{LO}}}}, \qquad r_0 = a_0\sqrt{\frac{E_{\text{Ry}}}{m_b\,\hbar\omega_{\text{LO}}}}$$
   where $\varepsilon_\infty$ is the high-frequency optical dielectric constant and $\varepsilon_0$ is the static dielectric constant ($\frac{1}{\tilde{\varepsilon}} = \frac{1}{\varepsilon_\infty} - \frac{1}{\varepsilon_0}$ is the Pekar ionic screening factor).

2. **Weak-Coupling Fröhlich Perturbation Theory ($\alpha < 6$) (`evaluate_frohlich_weak_coupling` — TODO 2)**:
   At weak coupling ($\alpha \ll 6$), second-order Rayleigh–Schrödinger perturbation theory (plus Röseler's fourth-order $\alpha^2$ coefficient $c_2 = 0.0159196$) yields:
   $$E_{\text{pol}}^{\text{weak}}(\alpha) = -\left(\alpha + 0.0159196\,\alpha^2\right)\hbar\omega_{\text{LO}}, \qquad \frac{m_{\text{weak}}^*}{m_b} = \frac{1}{1 - \alpha/6}, \qquad \langle N_{\text{ph}} \rangle_{\text{weak}} = \frac{\alpha}{2}$$
   Notice that the perturbative effective mass diverges at $\alpha = 6$, signaling the breakdown of weak-coupling perturbation theory and the onset of self-trapping!

3. **Strong-Coupling Landau–Pekar Variational Theory ($\alpha \gg 6$) (`evaluate_landau_pekar_strong_coupling` — TODO 3)**:
   In the strong-coupling regime, Landau and Pekar introduced a product variational ansatz $|\Psi\rangle = |\psi_{\text{el}}\rangle|\Phi_{\text{ph}}\rangle$ with a localized 3D Gaussian electron density $|\psi_{\text{el}}(\mathbf{r})|^2 = (\pi a_P^2)^{-3/2}e^{-r^2/a_P^2}$. Minimizing the total energy with respect to the localization radius $a_P$ gives ([arXiv:2109.12594](https://arxiv.org/html/2109.12594v1)):
   $$a_P(\alpha) = \frac{3\sqrt{\pi}}{\alpha}\,r_0, \qquad E_{\text{kin}} = \frac{\alpha^2}{3\pi}\hbar\omega_{\text{LO}}, \qquad E_{\text{pot}} = -\frac{2\alpha^2}{3\pi}\hbar\omega_{\text{LO}}, \qquad E_{\text{pol}}^{\text{LP}}(\alpha) = E_{\text{kin}} + E_{\text{pot}} = -\frac{\alpha^2}{3\pi}\hbar\omega_{\text{LO}}$$
   and the Landau–Pekar effective mass grows quartically with $\alpha$:
   $$\frac{m_{\text{LP}}^*}{m_b} = 1 + \frac{16\,\alpha^4}{81\pi^2}$$
   Comparing the leading weak-coupling energy $-\alpha\hbar\omega_{\text{LO}}$ and strong-coupling energy $-\frac{\alpha^2}{3\pi}\hbar\omega_{\text{LO}}$ defines the analytical crossover coupling $\alpha_{\text{cross}} = 3\pi \approx 9.4248$.

In [ ]:
def compute_frohlich_alpha_and_length(
    m_band_eff: float,
    omega_lo_ev: float,
    eps_inf: float,
    eps_static: float,
) -> dict[str, float]:
    '''
    TODO 1: Compute the Pekar factor (1/eps_inf - 1/eps_static), the dimensionless Fröhlich
    coupling constant `alpha`, and the characteristic polaron length `r0_ang` (in Angstrom).
    Return a dict with keys 'pekar_factor', 'alpha', and 'r0_ang'.
    '''
    # TODO 1: Use RYDBERG_EV and BOHR_TO_ANG to evaluate pekar_factor, alpha, and r0_ang.
    raise NotImplementedError("Complete TODO 1: compute_frohlich_alpha_and_length")


def evaluate_frohlich_weak_coupling(
    alpha: float, omega_lo_ev: float
) -> dict[str, float | bool]:
    '''
    TODO 2: Evaluate weak-coupling Fröhlich perturbation theory at coupling `alpha`:
    - 'e_pol_ev': -(alpha + 0.0159196 * alpha^2) * omega_lo_ev
    - 'm_eff_ratio': 1.0 / (1.0 - alpha / 6.0) if alpha < 6.0 else np.inf
    - 'mean_phonon_number': 0.5 * alpha
    - 'perturbation_valid': bool(alpha < 6.0)
    Return a dict with those 4 keys.
    '''
    # TODO 2: Implement the weak-coupling energy, effective mass ratio, mean phonon count, and validity flag.
    raise NotImplementedError("Complete TODO 2: evaluate_frohlich_weak_coupling")


def evaluate_landau_pekar_strong_coupling(
    alpha: float, omega_lo_ev: float, r0_ang: float
) -> dict[str, float]:
    '''
    TODO 3: Evaluate the strong-coupling Landau-Pekar Gaussian variational equations:
    - 'localization_radius_ang': (3 * sqrt(pi) / alpha) * r0_ang
    - 'e_kin_ev': (alpha^2 / (3 * pi)) * omega_lo_ev
    - 'e_pot_ev': -(2 * alpha^2 / (3 * pi)) * omega_lo_ev
    - 'e_pol_lp_ev': -(alpha^2 / (3 * pi)) * omega_lo_ev
    - 'm_eff_lp_ratio': 1.0 + (16.0 * alpha^4) / (81.0 * pi^2)
    - 'alpha_crossover': 3.0 * pi
    Return a dict with those 6 keys.
    '''
    # TODO 3: Evaluate all 6 strong-coupling Landau-Pekar quantities and return in a dictionary.
    raise NotImplementedError("Complete TODO 3: evaluate_landau_pekar_strong_coupling")

## Part 2: Lattice Holstein Model & Exact Diagonalization Across the Weak-to-Strong Crossover (TODOs 4–7)

### Mathematical Specifications
In molecular crystals, organic semiconductors, and narrow-band transition-metal oxides, short-range intramolecular vibrations couple locally to the electron density on each site via the **Holstein Hamiltonian** ([Phys. Rev. X 10, 021062 (2020)](https://link.aps.org/doi/10.1103/PhysRevX.10.021062); [arXiv:2603.07290](https://arxiv.org/html/2603.07290v2)):

$$\hat{H}_{\text{Holstein}} = -J\sum_{\langle i,j\rangle}\left(\hat{c}_i^\dagger\hat{c}_j + \hat{c}_j^\dagger\hat{c}_i\right) + \omega_0\sum_i \hat{b}_i^\dagger\hat{b}_i + g\omega_0\sum_i \hat{n}_i\left(\hat{b}_i + \hat{b}_i^\dagger\right) \qquad (\hbar = 1)$$

1. **Dimensionless Holstein Parameters & Regime Classification (`compute_holstein_parameters` — TODO 4)**:
   - Single-site polaron relaxation (binding) energy: $E_p = g^2\omega_0$
   - Bare electronic half-bandwidth in $d$ spatial dimensions: $W_0 = 2 d J$
   - Dimensionless polaron coupling: $\lambda = \frac{E_p}{W_0} = \frac{g^2\omega_0}{2 d J}$
   - Huang–Rhys factor: $S_{\text{HR}} = g^2$
   - Adiabaticity ratio: $\gamma_{\text{ad}} = \frac{\omega_0}{J}$
   - Regime classification: `"weak_coupling"` if ($g^2 < 0.5$ and $\lambda < 0.5$), `"strong_coupling"` if ($g^2 > 1.5$ or $\lambda > 1.0$), else `"intermediate_crossover"`.

2. **Exact Two-Site Holstein Dimer Hamiltonian (`build_holstein_dimer_hamiltonian` — TODO 5)**:
   For a two-site Holstein dimer ($i \in \{0, 1\}$, single electron in basis $\{|0\rangle_{\text{el}}, |1\rangle_{\text{el}}\}$), introducing the center-of-mass phonon mode $\hat{B}_+ = \frac{\hat{b}_0 + \hat{b}_1}{\sqrt{2}}$ (which couples only to the constant total electron number $\hat{n}_0 + \hat{n}_1 = 1$ and decouples from the dynamics!) and the **antisymmetric relative reaction-coordinate phonon mode** $\hat{b} = \frac{\hat{b}_0 - \hat{b}_1}{\sqrt{2}}$ reduces the exact dimer Hamiltonian to a $2 N_{\text{ph}} \times 2 N_{\text{ph}}$ matrix on $\mathcal{H}_{\text{el}}(2) \otimes \mathcal{H}_{\text{ph}}^{\text{rel}}(N_{\text{ph}})$ (plus the exact constant symmetric-mode relaxation shift $-\frac{1}{2}g^2\omega_0\,\mathbf{I}$):
   $$\hat{H}_{\text{dimer}} = -J\,\hat{\sigma}_x \otimes \hat{I}_{\text{ph}} + \hat{I}_{\text{el}} \otimes \left(\omega_0 \hat{b}^\dagger\hat{b} - \frac{1}{2}g^2\omega_0\hat{I}_{\text{ph}}\right) + \frac{g\omega_0}{\sqrt{2}}\,\hat{\sigma}_z \otimes \left(\hat{b} + \hat{b}^\dagger\right)$$
   where $\hat{\sigma}_x = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$ hops the electron between sites $0$ and $1$, and $\hat{\sigma}_z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$ measures site polarization $\hat{n}_0 - \hat{n}_1$!

3. **Analytical Weak-Coupling vs. Strong-Coupling Asymptotics (`evaluate_dimer_weak_and_strong_limits` — TODO 6)**:
   - **Weak-Coupling Perturbation Theory ($g \ll 1$)**: Treating $\hat{H}_{\text{ep}} = \frac{g\omega_0}{\sqrt{2}}\hat{\sigma}_z(\hat{b} + \hat{b}^\dagger)$ as a second-order perturbation on the bonding electronic state $|+\rangle_{\text{el}}\otimes|0\rangle_{\text{ph}}$ (energy $-J$, which couples to the antibonding state $|-\rangle_{\text{el}}\otimes|1\rangle_{\text{ph}}$ at energy $+J + \omega_0$, denominator $\Delta E = 2J + \omega_0$):
     $$E_0^{\text{weak}}(g) = -J - \frac{1}{2}g^2\omega_0 - \frac{(g\omega_0/\sqrt{2})^2}{2J + \omega_0} = -J - \frac{1}{2}g^2\omega_0\left(1 + \frac{\omega_0}{2J + \omega_0}\right)$$
     and the perturbatively narrowed tunneling splitting is:
     $$\Delta E_{+-}^{\text{weak}}(g) = 2J\left[1 - \frac{g^2\omega_0^2}{(2J + \omega_0)^2}\right]$$
   - **Strong-Coupling Lang–Firsov Theory ($g \gg 1$)**: Displacing the relative phonon mode by $\pm\frac{g}{\sqrt{2}}$ gives the exact localized polaron energy $-g^2\omega_0$ and exponentially narrowed Franck–Condon tunneling splitting:
     $$E_0^{\text{strong}}(g) = -g^2\omega_0 - J\,e^{-g^2}, \qquad \Delta E_{+-}^{\text{strong}}(g) = 2 J\,e^{-g^2}$$

4. **Exact Crossover Scan (`scan_holstein_crossover_curve` — TODO 7)**:
   Diagonalize $\hat{H}_{\text{dimer}}(g)$ across a grid of couplings $g$ to extract the exact ground-state energy $E_0(g)$, tunneling splitting $\Delta E_{+-}(g) = E_1(g) - E_0(g)$, and free-electron quasiparticle weight $Z(g) = |\langle +, 0_{\text{ph}} | \Psi_0(g) \rangle|^2$ (incorporating the symmetric-mode Franck–Condon overlap $e^{-g^2/2}$).

In [ ]:
def compute_holstein_parameters(
    j_hop: float, omega_0: float, g_ep: float, dim: int = 1
) -> dict[str, float | str]:
    '''
    TODO 4: Compute Holstein polaron parameters:
    - 'e_p': g_ep^2 * omega_0
    - 'w_half_bandwidth': 2 * dim * j_hop
    - 'lambda_ratio': e_p / w_half_bandwidth
    - 'huang_rhys_s': g_ep^2
    - 'adiabaticity_gamma': omega_0 / j_hop
    - 'regime': "weak_coupling" if (g_ep^2 < 0.5 and lambda_ratio < 0.5)
                else ("strong_coupling" if (g_ep^2 > 1.5 or lambda_ratio > 1.0) else "intermediate_crossover")
    Return a dict with those 6 keys.
    '''
    # TODO 4: Compute all 5 numerical ratios and the regime classification string.
    raise NotImplementedError("Complete TODO 4: compute_holstein_parameters")


def build_holstein_dimer_hamiltonian(
    j_hop: float, omega_0: float, g_ep: float, n_ph_cutoff: int = 16
) -> np.ndarray:
    '''
    TODO 5: Construct the (2 * n_ph_cutoff, 2 * n_ph_cutoff) Holstein dimer Hamiltonian matrix
    in the basis {|0>_el, |1>_el} (x) {|0>_ph, ..., |n_ph_cutoff - 1>_ph} using the relative
    phonon mode (including the exact symmetric-mode relaxation shift -0.5 * g_ep^2 * omega_0 * I).
    '''
    # TODO 5: Build phonon ladder operators b, b_dag, n_ph of size n_ph_cutoff, 2x2 Pauli matrices
    #         sigma_x, sigma_z, I_el, and assemble H_dimer via np.kron.
    raise NotImplementedError("Complete TODO 5: build_holstein_dimer_hamiltonian")


def evaluate_dimer_weak_and_strong_limits(
    j_hop: float, omega_0: float, g_ep: float
) -> dict[str, float]:
    '''
    TODO 6: Evaluate the analytical weak-coupling perturbation theory and strong-coupling Lang-Firsov
    predictions for the Holstein dimer ground-state energy and tunneling splitting:
    - 'e0_weak': -j_hop - 0.5 * g_ep^2 * omega_0 * (1.0 + omega_0 / (2.0 * j_hop + omega_0))
    - 'splitting_weak': 2.0 * j_hop * (1.0 - (g_ep * omega_0 / (2.0 * j_hop + omega_0))**2)
    - 'e0_strong': -g_ep^2 * omega_0 - j_hop * exp(-g_ep^2)
    - 'splitting_strong': 2.0 * j_hop * exp(-g_ep^2)
    Return a dict with those 4 keys.
    '''
    # TODO 6: Compute and return the 4 analytical asymptotic expressions.
    raise NotImplementedError("Complete TODO 6: evaluate_dimer_weak_and_strong_limits")


def scan_holstein_crossover_curve(
    j_hop: float, omega_0: float, g_grid: np.ndarray, n_ph_cutoff: int = 18
) -> dict[str, np.ndarray]:
    '''
    TODO 7: For each coupling `g` in `g_grid`, diagonalize `build_holstein_dimer_hamiltonian(j_hop, omega_0, g, n_ph_cutoff)`
    and evaluate `evaluate_dimer_weak_and_strong_limits(j_hop, omega_0, g)`.
    Also compute the quasiparticle weight Z(g) = exp(-0.5 * g^2) * |<+, 0_rel | Psi_0(g)>|^2, where
    |+, 0_rel> = ([1/sqrt(2), 1/sqrt(2)] (x) [1, 0, ..., 0]).
    Return a dict with 1D arrays of length len(g_grid):
    'g_grid', 'e0_exact', 'e0_weak', 'e0_strong', 'splitting_exact', 'splitting_weak', 'splitting_strong', 'quasiparticle_z'.
    '''
    # TODO 7: Loop over g_grid, diagonalize H_dimer with eigh, and record all 7 curves.
    raise NotImplementedError("Complete TODO 7: scan_holstein_crossover_curve")

## Part 3: Strong-Coupling Lang–Firsov Transformation, Band Narrowing & Poisson Spectral Function (TODOs 8–11)

### Mathematical Specifications
1. **Unitary Phonon Displacement Matrix $\hat{D}(g) = \exp\!\big[g(\hat{b}^\dagger - \hat{b})\big]$ (`build_displacement_operator_matrix` — TODO 8)**:
   In the $N_{\text{ph}}$-dimensional truncated Fock basis, construct $\hat{D}(g) = \exp\!\big[g(\hat{b}^\dagger - \hat{b})\big]$ via `scipy.linalg.expm` and verify that the coherent-state Franck–Condon overlap squared $|\langle n | \hat{D}(g) | 0 \rangle|^2$ equals the Poisson distribution $P_n(g^2) = e^{-g^2}\frac{g^{2n}}{n!}$!

2. **Thermal Lang–Firsov Bandwidth Renormalization & Effective Mass (`compute_lang_firsov_bandwidth_and_mass` — TODO 9)** ([Phys. Rev. X 10, 021062 (2020)](https://link.aps.org/doi/10.1103/PhysRevX.10.021062)):
   At thermal energy $k_B T$, the Bose–Einstein phonon occupation is $n_{\text{BE}}(\omega_0, T) = \frac{1}{e^{\omega_0 / k_B T} - 1}$ (for $k_B T > 0$, and $0$ at $T = 0$). Averaging the two-site displacement operator $\hat{D}_i^\dagger(g)\hat{D}_j(g)$ over the thermal phonon bath renormalizes the hopping integral $J \to \tilde{J}(T)$ and enhances the band effective mass $m^*/m_0$:
   $$\tilde{J}(T) = J\,\exp\!\left[-g^2\left(2\,n_{\text{BE}}(\omega_0, T) + 1\right)\right], \qquad \frac{m^*(T)}{m_0} = \frac{J}{\tilde{J}(T)} = \exp\!\left[+g^2\left(2\,n_{\text{BE}}(\omega_0, T) + 1\right)\right]$$

3. **1D Bare vs. Polaronic Band Structure (`compute_1d_polaron_band_dispersion` — TODO 10)**:
   For wavevectors $k a \in [-\pi, +\pi]$:
   $$E_{\text{bare}}(k) = -2 J \cos(ka), \qquad E_{\text{pol}}(k, T) = -g^2\omega_0 - 2\tilde{J}(T)\cos(ka)$$

4. **Strong-Coupling Polaron Spectral Function $A(\omega)$ (`compute_polaron_spectral_function` — TODO 11)**:
   In the localized strong-coupling limit ($T = 0$), the single-particle electron-addition spectral function exhibits a progression of Lorentzian/Gaussian **phonon sidebands (replicas)** spaced by $\omega_0$ above the polaron ground state $\omega_{\text{ZPL}} = -g^2\omega_0$:
   $$A(\omega) = \sum_{n=0}^{N_{\max}} e^{-g^2}\frac{g^{2n}}{n!}\,\frac{1}{\sqrt{2\pi}\sigma}\exp\!\left(-\frac{\big[\omega - (-g^2\omega_0 + n\omega_0)\big]^2}{2\sigma^2}\right)$$

In [ ]:
def build_displacement_operator_matrix(
    g_ep: float, n_ph_cutoff: int = 20
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''
    TODO 8: Construct the (n_ph_cutoff, n_ph_cutoff) unitary phonon displacement operator
    D(g) = expm(g * (b_dag - b)), extract the numerical vacuum Franck-Condon weights
    fc_num[n] = |D[n, 0]|^2, and compute the analytical Poisson weights fc_poisson[n] = exp(-g^2) * g^(2n) / n!.
    Return (D_mat, fc_num, fc_poisson).
    '''
    # TODO 8: Build b and b_dag of size n_ph_cutoff, compute D_mat = expm(g_ep * (b_dag - b)),
    #         fc_num = np.abs(D_mat[:, 0])**2, and fc_poisson using scipy.special.factorial.
    raise NotImplementedError("Complete TODO 8: build_displacement_operator_matrix")


def compute_lang_firsov_bandwidth_and_mass(
    j_hop: float, g_ep: float, omega_0: float, temperature_kt: float = 0.0
) -> dict[str, float]:
    '''
    TODO 9: Compute the Bose-Einstein thermal occupation `n_be`, the Franck-Condon narrowing factor
    `fc_factor = exp(-g_ep^2 * (2*n_be + 1))`, the renormalized hopping `j_eff = j_hop * fc_factor`,
    the polaron bandwidth `w_pol = 4 * j_eff` (in 1D), and the effective mass ratio `m_eff_ratio = 1 / fc_factor`.
    Return a dict with keys 'n_be', 'fc_factor', 'j_eff', 'w_pol', 'm_eff_ratio'.
    '''
    # TODO 9: Handle temperature_kt <= 1e-12 (n_be = 0.0) vs temperature_kt > 1e-12, and compute all 5 quantities.
    raise NotImplementedError("Complete TODO 9: compute_lang_firsov_bandwidth_and_mass")


def compute_1d_polaron_band_dispersion(
    ka_grid: np.ndarray, j_hop: float, g_ep: float, omega_0: float, temperature_kt: float = 0.0
) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 10: Evaluate the 1D bare band E_bare(ka) = -2 * j_hop * cos(ka) and the Lang-Firsov
    polaron band E_pol(ka) = -g_ep^2 * omega_0 - 2 * j_eff(T) * cos(ka) on `ka_grid`.
    Return (E_bare, E_pol).
    '''
    # TODO 10: Call compute_lang_firsov_bandwidth_and_mass and evaluate E_bare and E_pol on ka_grid.
    raise NotImplementedError("Complete TODO 10: compute_1d_polaron_band_dispersion")


def compute_polaron_spectral_function(
    omega_grid: np.ndarray,
    g_ep: float,
    omega_0: float,
    n_replicas: int = 12,
    sigma_broad: float = 0.08,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''
    TODO 11: Compute the T=0 strong-coupling polaron spectral function A(omega) on `omega_grid`,
    along with the replica peak frequencies `peak_freqs` (-g^2*omega_0 + n*omega_0 for n=0..n_replicas-1)
    and Poisson weights `poisson_weights` (exp(-g^2) * g^(2n) / n!).
    Return (A_omega, peak_freqs, poisson_weights).
    '''
    # TODO 11: Sum Gaussian peaks centered at peak_freqs[n] weighted by poisson_weights[n].
    raise NotImplementedError("Complete TODO 11: compute_polaron_spectral_function")

## Part 4: Real-Time Polaron Dynamics, Coherent-to-Incoherent Transport Crossover & Ehrenfest Self-Trapping (TODOs 12–15)

### Mathematical Specifications
1. **Real-Time Quantum Evolution of a Holstein Dimer (`propagate_holstein_dimer_dynamics` — TODO 12)**:
   Starting with the electron localized on Site $0$ and the relative phonon mode in its vacuum state $|\Psi(0)\rangle = |0\rangle_{\text{el}} \otimes |0\rangle_{\text{ph}}$, evolve $|\Psi(t)\rangle = e^{-i\hat{H}_{\text{dimer}} t}|\Psi(0)\rangle$ using eigendecomposition $\hat{H}_{\text{dimer}} = \mathbf{V}\text{diag}(\boldsymbol{\varepsilon})\mathbf{V}^\dagger$ and track:
   - Site-0 electronic population $P_0(t) = \langle \Psi(t) | (\hat{P}_{0,\text{el}} \otimes \hat{I}_{\text{ph}}) | \Psi(t) \rangle$
   - Relative phonon excitation number $\langle \hat{n}_{\text{ph}}(t) \rangle = \langle \Psi(t) | (\hat{I}_{\text{el}} \otimes \hat{b}^\dagger\hat{b}) | \Psi(t) \rangle$
   Compare $g = 0.25$ (**weak coupling**: fast, nearly undamped Rabi oscillations at frequency $2J$) against $g = 1.8$ (**strong coupling**: immediate phonon emission followed by **self-trapping** on Site 0 over short timescales!).

2. **Coherent Band Tunneling vs. Incoherent Marcus Hopping Mobility Crossover (`compute_band_vs_hopping_crossover` — TODO 13)** ([Janković PRB 2022](https://www.scl.rs/papers/Jankovic_PRB_2022.pdf); [Phys. Rev. X 10, 021062](https://link.aps.org/doi/10.1103/PhysRevX.10.021062)):
   Across a temperature grid `temperature_kt_grid`:
   - **Coherent band-like transfer rate**: $k_{\text{coh}}(T) = \frac{2\,\tilde{J}(T)^2}{\gamma_{\text{dephase}}}$ (decreases with $T$ as thermal phonons narrow the polaron band $\tilde{J}(T)$).
   - **Incoherent thermally activated Marcus / small-polaron hopping rate** (activation barrier $E_a = \frac{E_p}{2} = \frac{1}{2}g^2\omega_0$):
     $$k_{\text{hop}}(T) = J^2 \sqrt{\frac{\pi}{2 E_p k_B T}}\,\exp\!\left(-\frac{E_p}{2 k_B T}\right)$$
   - **Einstein Charge Mobility ($\mu \propto D / k_B T$)**:
     $$\mu_{\text{coh}}(T) = \frac{k_{\text{coh}}(T)}{k_B T}, \qquad \mu_{\text{hop}}(T) = \frac{k_{\text{hop}}(T)}{k_B T}, \qquad \mu_{\text{tot}}(T) = \mu_{\text{coh}}(T) + \mu_{\text{hop}}(T)$$

3. **Holstein (Diagonal) vs. SSH/Peierls (Off-Diagonal) Coupling Comparison (`compare_holstein_vs_ssh_peierls_effective_coupling` — TODO 14)** ([Phys. Rev. B 106, 035133 (2022)](https://link.aps.org/doi/10.1103/PhysRevB.106.035133)):
   In momentum space, the electron–phonon vertex $g(k, q)$ for the **Holstein** model is momentum-independent ($|g_{\text{Hol}}(q)|^2 = g^2\omega_0^2$), whereas for the **Su–Schrieffer–Heeger (SSH) / Peierls** model (where lattice displacements modulate the nearest-neighbor hopping integral) the coupling vertex depends on both electron momentum $k$ and phonon wavevector $q$:
   $$|g_{\text{SSH}}(k, q)|^2 = 4\alpha_{\text{SSH}}^2\big[\sin(k + q) - \sin(k)\big]^2$$
   Notice that at the band minimum ($k = 0$) and long-wavelength acoustic limit ($q \to 0$), $|g_{\text{SSH}}(0, 0)|^2 = 0$, explaining why Peierls polarons remain much lighter than Holstein polarons!

4. **Mixed Quantum–Classical Ehrenfest Polaron Self-Trapping on a 1D Chain (`run_ehrenfest_polaron_chain` — TODO 15)**:
   On an $L$-site chain with classical dimensionless oscillator displacements $q_j(t)$ and velocities $v_j(t) = \dot{q}_j(t)$ coupled to a quantum electronic statevector $\mathbf{c}(t)$ ($\sum_j |c_j|^2 = 1$):
   $$i\dot{c}_j = -J\left(c_{j-1} + c_{j+1}\right) + \sqrt{2}\,g\omega_0\,q_j\,c_j, \qquad \ddot{q}_j = -\omega_0^2\,q_j - \sqrt{2}\,g\omega_0^2\,|c_j|^2$$
   Compute the **Inverse Participation Ratio** $\text{IPR}(t) = \sum_{j=0}^{L-1} |c_j(t)|^4 \in [1/L, 1]$, which equals $1/L$ for a completely delocalized Bloch wave (weak coupling) and approaches $1$ when the polaron **self-traps** on a single site (strong coupling)!

In [ ]:
def propagate_holstein_dimer_dynamics(
    j_hop: float,
    omega_0: float,
    g_ep: float,
    n_ph_cutoff: int = 14,
    dt: float = 0.05,
    n_steps: int = 160,
) -> dict[str, np.ndarray]:
    '''
    TODO 12: Propagate the Holstein dimer starting from |Psi(0)> = |0>_el (x) |0>_ph (electron on site 0,
    relative phonon mode in vacuum) over `n_steps` steps of size `dt` using exact spectral decomposition
    of `build_holstein_dimer_hamiltonian`.
    Return a dict with keys 'times', 'pop_site0', and 'mean_rel_phonons'.
    '''
    # TODO 12: Diagonalize H_dimer = V @ diag(evals) @ V.T, evolve psi(t) = V @ (exp(-1j * evals * t) * (V.T @ psi_0)),
    #          and compute pop_site0(t) and <n_ph(t)> at each step.
    raise NotImplementedError("Complete TODO 12: propagate_holstein_dimer_dynamics")


def compute_band_vs_hopping_crossover(
    j_hop: float,
    omega_0: float,
    g_ep: float,
    temperature_kt_grid: np.ndarray,
    gamma_dephase: float = 0.02,
) -> dict[str, np.ndarray | float]:
    '''
    TODO 13: Across `temperature_kt_grid`, evaluate:
    - 'k_coh': 2 * j_eff(T)^2 / gamma_dephase
    - 'k_hop': j_hop^2 * sqrt(pi / (2 * e_p * kT)) * exp(-e_p / (2 * kT)) (where e_p = g_ep^2 * omega_0)
    - 'mu_coh': k_coh / kT
    - 'mu_hop': k_hop / kT
    - 'mu_tot': mu_coh + mu_hop
    - 't_min_mobility': float(temperature_kt_grid[argmin(mu_tot)])
    Return a dict with those 6 keys.
    '''
    # TODO 13: Loop over or vectorize across temperature_kt_grid using compute_lang_firsov_bandwidth_and_mass.
    raise NotImplementedError("Complete TODO 13: compute_band_vs_hopping_crossover")


def compare_holstein_vs_ssh_peierls_vertex(
    k_elec: float, q_grid: np.ndarray, g_holstein: float, omega_0: float, alpha_ssh: float
) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 14: Evaluate the squared electron-phonon coupling vertex |g(k, q)|^2 across phonon wavevectors `q_grid`
    for:
    1. Holstein coupling: v_sq_holstein = full_like(q_grid, (g_holstein * omega_0)**2)
    2. SSH / Peierls coupling: v_sq_ssh = 4.0 * (alpha_ssh**2) * (sin(k_elec + q_grid) - sin(k_elec))**2
    Return (v_sq_holstein, v_sq_ssh).
    '''
    # TODO 14: Compute and return (v_sq_holstein, v_sq_ssh) on q_grid.
    raise NotImplementedError("Complete TODO 14: compare_holstein_vs_ssh_peierls_vertex")


def run_ehrenfest_polaron_chain(
    n_sites: int = 11,
    j_hop: float = 1.0,
    omega_0: float = 0.5,
    g_ep: float = 1.8,
    dt: float = 0.04,
    n_steps: int = 150,
) -> dict[str, np.ndarray | float]:
    '''
    TODO 15: Propagate mixed quantum-classical Ehrenfest polaron dynamics on an `n_sites` open chain.
    Initialize the electron on the center site `c[n_sites // 2] = 1.0` with a small initial lattice
    seed displacement `q[n_sites // 2] = -sqrt(2) * g_ep` (equilibrium displacement for a localized carrier)
    and `v[:] = 0.0`. At each step of size `dt`:
    - Advance (q, v) via Velocity Verlet under force F_j(q, c) = -omega_0^2 * q_j - sqrt(2) * g_ep * omega_0^2 * |c_j|^2
    - Advance `c` via exact matrix exponential expm(-1j * H_el(q_mid) * dt), where
      H_el(q) has -j_hop on first off-diagonals and +sqrt(2) * g_ep * omega_0 * q_j on the diagonal.
    Track the Inverse Participation Ratio IPR(t) = sum_j |c_j(t)|^4.
    Return a dict with keys 'times', 'ipr_history', 'final_site_pops', and 'time_averaged_ipr'.
    '''
    # TODO 15: Integrate the coupled Ehrenfest equations over n_steps and return the result dictionary.
    raise NotImplementedError("Complete TODO 15: run_ehrenfest_polaron_chain")

## Part 5: Step-by-Step Checkpoint & Verification Suite

Run `run_section07b_polaron_checkpoints()` below as you implement each `# TODO` function to test your continuum Fröhlich weak/strong solvers, exact Holstein dimer crossover diagonalizer, Lang–Firsov displacement operator and band narrowing, spectral function replicas, band-to-hopping transport crossover, and Ehrenfest self-trapping simulator!

In [2]:
def run_section07b_polaron_checkpoints() -> None:
    '''Step-by-step checkpoint verifier for all 15 #TODO functions in Section 07b (Polaron Theory & Simulation).'''
    passed = 0
    total_todos = 15

    try:
        # --- PART 1 CHECKS: CONTINUUM FROHLICH POLARONS (TODOs 1-3) ---
        fr_gaas = compute_frohlich_alpha_and_length(m_band_eff=0.067, omega_lo_ev=0.036, eps_inf=10.9, eps_static=12.9)
        fr_naf = compute_frohlich_alpha_and_length(m_band_eff=1.0, omega_lo_ev=0.050, eps_inf=1.74, eps_static=5.10)
        assert 0.05 < fr_gaas["alpha"] < 0.10 and fr_naf["alpha"] > 5.0 and fr_gaas["r0_ang"] > 20.0
        passed += 1
        print(f"[PASS] TODO 1 : compute_frohlich_alpha_and_length (GaAs alpha = {fr_gaas['alpha']:.4f}, NaF alpha = {fr_naf['alpha']:.3f})")

        wk_gaas = evaluate_frohlich_weak_coupling(fr_gaas["alpha"], 0.036)
        wk_bad = evaluate_frohlich_weak_coupling(6.5, 0.036)
        assert wk_gaas["perturbation_valid"] and wk_gaas["m_eff_ratio"] > 1.0 and not wk_bad["perturbation_valid"]
        passed += 1
        print(f"[PASS] TODO 2 : evaluate_frohlich_weak_coupling (GaAs m*/m_b = {wk_gaas['m_eff_ratio']:.5f})")

        lp_strong = evaluate_landau_pekar_strong_coupling(10.0, 0.050, 10.0)
        assert np.isclose(lp_strong["e_pol_lp_ev"], -(100.0 / (3.0 * np.pi)) * 0.050)
        assert np.isclose(lp_strong["e_kin_ev"] + lp_strong["e_pot_ev"], lp_strong["e_pol_lp_ev"])
        assert np.isclose(lp_strong["alpha_crossover"], 3.0 * np.pi)
        passed += 1
        print(f"[PASS] TODO 3 : evaluate_landau_pekar_strong_coupling (alpha=10 m*/m_b = {lp_strong['m_eff_lp_ratio']:.2f})")

        # --- PART 2 CHECKS: LATTICE HOLSTEIN CROSSOVER (TODOs 4-7) ---
        hp_wk = compute_holstein_parameters(j_hop=1.0, omega_0=1.0, g_ep=0.3, dim=1)
        hp_st = compute_holstein_parameters(j_hop=1.0, omega_0=1.0, g_ep=1.8, dim=1)
        assert hp_wk["regime"] == "weak_coupling" and hp_st["regime"] == "strong_coupling"
        passed += 1
        print("[PASS] TODO 4 : compute_holstein_parameters")

        H_dim_0 = build_holstein_dimer_hamiltonian(j_hop=1.0, omega_0=1.0, g_ep=0.0, n_ph_cutoff=10)
        evals_0 = np.sort(np.linalg.eigvalsh(H_dim_0))
        assert H_dim_0.shape == (20, 20) and np.isclose(evals_0[0], -1.0) and np.isclose(evals_0[1], 0.0)
        passed += 1
        print("[PASS] TODO 5 : build_holstein_dimer_hamiltonian")

        lim_wk = evaluate_dimer_weak_and_strong_limits(1.0, 1.0, 0.15)
        lim_st = evaluate_dimer_weak_and_strong_limits(1.0, 1.0, 2.50)
        H_wk = build_holstein_dimer_hamiltonian(1.0, 1.0, 0.15, n_ph_cutoff=16)
        H_st = build_holstein_dimer_hamiltonian(1.0, 1.0, 2.50, n_ph_cutoff=22)
        ev_wk = np.sort(np.linalg.eigvalsh(H_wk))
        ev_st = np.sort(np.linalg.eigvalsh(H_st))
        assert np.isclose(ev_wk[0], lim_wk["e0_weak"], atol=5e-4)
        assert np.isclose(ev_st[0], lim_st["e0_strong"], atol=0.03)
        passed += 1
        print(f"[PASS] TODO 6 : evaluate_dimer_weak_and_strong_limits (g=0.15 exact={ev_wk[0]:.5f} vs weak={lim_wk['e0_weak']:.5f})")

        scan = scan_holstein_crossover_curve(1.0, 1.0, np.linspace(0.0, 2.2, 12), n_ph_cutoff=18)
        assert np.isclose(scan["quasiparticle_z"][0], 1.0) and scan["quasiparticle_z"][-1] < 0.05
        assert scan["splitting_exact"][-1] < 0.1 * scan["splitting_exact"][0]
        passed += 1
        print(f"[PASS] TODO 7 : scan_holstein_crossover_curve (Z(0)={scan['quasiparticle_z'][0]:.3f} -> Z(2.2)={scan['quasiparticle_z'][-1]:.4f})")

        # --- PART 3 CHECKS: LANG-FIRSOV TRANSFORMATION & SPECTRAL REPLICAS (TODOs 8-11) ---
        D_mat, fc_num, fc_pois = build_displacement_operator_matrix(g_ep=1.2, n_ph_cutoff=24)
        assert np.linalg.norm(D_mat.conj().T @ D_mat - np.eye(24)) < 1e-12
        assert np.allclose(fc_num[:8], fc_pois[:8], atol=1e-4)
        passed += 1
        print("[PASS] TODO 8 : build_displacement_operator_matrix (Unitary D(g) matches Poisson Franck-Condon weights)")

        lf_0 = compute_lang_firsov_bandwidth_and_mass(j_hop=1.0, g_ep=1.5, omega_0=0.5, temperature_kt=0.0)
        lf_t = compute_lang_firsov_bandwidth_and_mass(j_hop=1.0, g_ep=1.5, omega_0=0.5, temperature_kt=0.5)
        assert np.isclose(lf_0["j_eff"], np.exp(-2.25)) and lf_t["j_eff"] < lf_0["j_eff"] and lf_t["m_eff_ratio"] > lf_0["m_eff_ratio"]
        passed += 1
        print(f"[PASS] TODO 9 : compute_lang_firsov_bandwidth_and_mass (T=0 J_eff/J = {lf_0['fc_factor']:.4f}, T=0.5 J_eff/J = {lf_t['fc_factor']:.4f})")

        ka = np.linspace(-np.pi, np.pi, 41)
        e_bare, e_pol = compute_1d_polaron_band_dispersion(ka, j_hop=1.0, g_ep=1.5, omega_0=0.5, temperature_kt=0.0)
        assert np.isclose(np.ptp(e_bare), 4.0) and np.isclose(np.ptp(e_pol), 4.0 * np.exp(-2.25))
        passed += 1
        print("[PASS] TODO 10: compute_1d_polaron_band_dispersion")

        w_grid = np.linspace(-3.0, 5.0, 800)
        A_w, p_freqs, p_weights = compute_polaron_spectral_function(w_grid, g_ep=1.5, omega_0=0.5, n_replicas=12, sigma_broad=0.06)
        dw = w_grid[1] - w_grid[0]
        assert np.isclose(np.sum(A_w) * dw, np.sum(p_weights), atol=1e-2) and np.argmax(p_weights) in (2, 3)
        passed += 1
        print(f"[PASS] TODO 11: compute_polaron_spectral_function (Integrated spectral weight = {np.sum(A_w)*dw:.4f})")

        # --- PART 4 CHECKS: REAL-TIME DYNAMICS, CROSSOVER & EHRENFEST SELF-TRAPPING (TODOs 12-15) ---
        dyn_wk = propagate_holstein_dimer_dynamics(1.0, 1.0, g_ep=0.2, n_ph_cutoff=12, dt=0.05, n_steps=80)
        dyn_st = propagate_holstein_dimer_dynamics(1.0, 1.0, g_ep=2.0, n_ph_cutoff=16, dt=0.05, n_steps=80)
        assert np.min(dyn_wk["pop_site0"]) < 0.15 and np.min(dyn_st["pop_site0"]) > 0.65
        passed += 1
        print(f"[PASS] TODO 12: propagate_holstein_dimer_dynamics (Weak min P_0 = {np.min(dyn_wk['pop_site0']):.3f} vs Strong self-trapped min P_0 = {np.min(dyn_st['pop_site0']):.3f})")

        kt_grid = np.linspace(0.08, 1.20, 50)
        trans = compute_band_vs_hopping_crossover(j_hop=0.15, omega_0=0.20, g_ep=1.6, temperature_kt_grid=kt_grid, gamma_dephase=0.01)
        assert trans["k_coh"][0] > trans["k_coh"][-1] and trans["k_hop"][-1] > trans["k_hop"][0]
        assert 0.08 < trans["t_min_mobility"] < 1.20
        passed += 1
        print(f"[PASS] TODO 13: compute_band_vs_hopping_crossover (Coherent-to-Hopping mobility minimum at kT = {trans['t_min_mobility']:.3f})")

        q_arr = np.linspace(-np.pi, np.pi, 51)
        v_hol, v_ssh = compare_holstein_vs_ssh_peierls_vertex(0.0, q_arr, g_holstein=1.2, omega_0=0.5, alpha_ssh=0.4)
        assert np.isclose(v_ssh[25], 0.0) and v_hol[25] > 0.0
        passed += 1
        print("[PASS] TODO 14: compare_holstein_vs_ssh_peierls_vertex (SSH vertex vanishes at q=0, k=0)")

        ehr_wk = run_ehrenfest_polaron_chain(n_sites=11, j_hop=1.0, omega_0=0.5, g_ep=0.2, dt=0.04, n_steps=120)
        ehr_st = run_ehrenfest_polaron_chain(n_sites=11, j_hop=1.0, omega_0=0.5, g_ep=2.0, dt=0.04, n_steps=120)
        assert ehr_st["time_averaged_ipr"] > 2.5 * ehr_wk["time_averaged_ipr"]
        passed += 1
        print(f"[PASS] TODO 15: run_ehrenfest_polaron_chain (Weak <IPR> = {ehr_wk['time_averaged_ipr']:.3f} vs Strong Self-Trapped <IPR> = {ehr_st['time_averaged_ipr']:.3f})")
        print(f"\nCONGRATULATIONS! All {passed}/{total_todos} Section 07b (Polaron Weak & Strong Coupling) checkpoints passed!")

    except NotImplementedError as exc:
        print(f"\n[PROGRESS: {passed}/{total_todos} checkpoints passed] Next up -> {exc}")
    except AssertionError as exc:
        print(f"\n[ASSERTION FAILED after {passed}/{total_todos} checkpoints] -> {exc}")


# Uncomment and run as you implement each TODO:
# run_section07b_polaron_checkpoints()
print("Section 07b Polaron checkpoint suite loaded. Implement TODOs 1-15 and call `run_section07b_polaron_checkpoints()`!")

Section 07b Polaron checkpoint suite loaded. Implement TODOs 1-15 and call `run_section07b_polaron_checkpoints()`!
